# Images 02 · Image-only, fusion and tabular baselines on a shared split (SUPERSEDED)

**Objective:** the first multimodal design. Compare an image-only CNN, a neural image + tabular fusion network and a tabular Random Forest on one shared 70/15/15 split, and show why this design could not support final claims.

**Derived from:** `notebooks/images/02_image_baseline.ipynb` (CNN and ResNet-18 baselines), `03_multimodal_data_alignment.ipynb` (shared split and fusion network), `04_image_only_shared_split.ipynb`, `05_tabular_only_shared_split.ipynb`.
**Design:** V1, a stratified **row-level** split, 706 train / 151 validation / 152 test, of the 1,009 K2 tissues.

> **SUPERSEDED, because:** (1) 108 of 239 fellow-eye pairs are split across partitions; (2) a single 152-tissue test set with no uncertainty estimate; (3) fusion is confounded with model type (a small neural network vs a Random Forest); (4) CNNs saw 224-pixel downscaled images. Replaced by frozen full-resolution embeddings, Random Forests and donor-grouped CV (Hybrid 01–04).

**Runtime:** under 1 minute by default. The neural networks need a GPU and are not deterministic, so they retrain only with `TRAIN_NEURAL_MODELS = True` (≈10 min on GPU; results will differ slightly from the recorded ones). When it is `False`, their results are the values recorded in the original notebooks' saved outputs.

In [1]:
TRAIN_NEURAL_MODELS = False

Set up paths and load the shared split saved by notebook images/03.

In [2]:
import copy
import random
import sys
import warnings
from pathlib import Path

GRADING_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "modeling_setup.py").is_file())
sys.path[:0] = [str(GRADING_DIR / "scripts"), str(GRADING_DIR / "notebooks_cleaned")]

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from torch.utils.data import DataLoader, Dataset

import cleaned_setup as cs
import modeling_setup as ms

warnings.filterwarnings("ignore", category=UserWarning)
shared = pd.read_csv(cs.SHARED_SPLIT_PATH)
pd.crosstab(shared["Split"], shared["Tissue_Grade_Clean"])[cs.CLASSES].reindex(["train", "validation", "test"])

Tissue_Grade_Clean,A+,A,B
Split,,,
train,371,220,115
validation,79,47,25
test,80,47,25


How many fellow-eye pairs does this split separate? (Donors inferred as in Hybrid 02.)

In [3]:
labelled = ms.load_labelled_data()
labelled["Donor_Group"] = ms.get_donor_groups(labelled)
assignment = labelled["Tissue_ID"].map(cs.normalize_id).map(dict(zip(shared["Normalized_ID"], shared["Split"])))
per_donor = pd.DataFrame({"donor": labelled["Donor_Group"], "split": assignment}).dropna().groupby("donor")["split"].agg(["size", "nunique"])
paired = per_donor[per_donor["size"] > 1]
print(f"Fellow-eye pairs in the split: {len(paired)}; split across partitions: {(paired['nunique'] > 1).sum()}")

Fellow-eye pairs in the split: 239; split across partitions: 108


Attach the 24 features and image paths to the split (same rows and order as notebooks images/03–05).

In [4]:
data = pd.read_excel(ms.DATA_PATH)
data["Normalized_ID"] = data["Tissue_ID"].map(cs.normalize_id)
data = data.merge(shared[["Normalized_ID", "Split"]], on="Normalized_ID", how="inner", validate="one_to_one")
manifest = pd.read_csv(cs.MANIFEST_PATH)
data = data.merge(manifest[["Normalized_ID", "Selected_Image"]], on="Normalized_ID", how="left", validate="one_to_one")
data["Image_Path"] = data["Selected_Image"].map(lambda p: cs.IMAGE_DIR / Path(p).name)
parts = {name: data[data["Split"] == name].reset_index(drop=True) for name in ["train", "validation", "test"]}
print({k: len(v) for k, v in parts.items()})

{'train': 706, 'validation': 151, 'test': 152}


## 1. Tabular Random Forest (notebook images/05)

Fit the preprocessing on the training rows only, pick a classifier by validation macro F1, and evaluate it once on the test rows.

In [5]:
pre = ms.make_preprocessor()
Z = {name: (pre.fit_transform if name == "train" else pre.transform)(part[ms.ALL_FEATURES]) for name, part in parts.items()}
y_split = {name: part[ms.TARGET] for name, part in parts.items()}
candidates = {
    "Logistic regression": LogisticRegression(max_iter=2000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    "Extra Trees": ExtraTreesClassifier(n_estimators=300, random_state=42, n_jobs=-1),
}
validation = {name: f1_score(y_split["validation"], m.fit(Z["train"], y_split["train"]).predict(Z["validation"]), average="macro")
              for name, m in candidates.items()}
best = max(validation, key=validation.get)
print("Validation macro F1:", {k: round(v, 3) for k, v in validation.items()}, "-> selected:", best)
test_pred = candidates[best].predict(Z["test"])
METRICS = {"Accuracy": accuracy_score, "Balanced accuracy": balanced_accuracy_score,
           "Macro F1": lambda t, p: f1_score(t, p, average="macro")}
results = {"Tabular Random Forest": {m: fn(y_split["test"], test_pred) for m, fn in METRICS.items()} | {"Source": "computed here"}}
pd.Series({m: round(v, 3) for m, v in results["Tabular Random Forest"].items() if m != "Source"}).to_frame("Test set (152 tissues)")

Validation macro F1: {'Logistic regression': 0.722, 'Random Forest': 0.836, 'Extra Trees': 0.684} -> selected: Random Forest


,Test set (152 tissues)
Accuracy,0.901
Balanced accuracy,0.893
Macro F1,0.906


## 2. Neural models (notebooks images/02–04)

Images are downscaled to fit 224 × 224 (aspect ratio kept, zero-padded) and scaled to [0, 1]. Each dataset item returns the image, the preprocessed tabular row (fusion only) and the grade index.

In [6]:
CLASS_NAMES = ["A", "A+", "B"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class CorneaDataset(Dataset):
    def __init__(self, rows, tabular):
        self.rows, self.tabular = rows.reset_index(drop=True), np.asarray(tabular, dtype=np.float32)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows.iloc[idx]
        with Image.open(row["Image_Path"]) as img:
            img = img.convert("L")
            img.thumbnail((224, 224), Image.Resampling.LANCZOS)
            canvas = Image.new("L", (224, 224), color=0)
            canvas.paste(img, ((224 - img.width) // 2, (224 - img.height) // 2))
        image = torch.from_numpy(np.asarray(canvas, dtype=np.float32)[None] / 255.0)
        return image, torch.from_numpy(self.tabular[idx]), CLASS_TO_IDX[row[ms.TARGET]]

The three architectures: a small CNN trained from scratch (23,715 parameters), the fusion network (the same image encoder + a tabular MLP, concatenated), and ImageNet ResNet-18 with a new 3-grade head.

In [7]:
def image_encoder():
    return nn.Sequential(
        nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
    )


class CorneaCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features, self.classifier = image_encoder(), nn.Sequential(nn.Dropout(0.3), nn.Linear(64, 3))

    def forward(self, images, tabular=None):
        return self.classifier(self.features(images))


class FusionNet(nn.Module):
    def __init__(self, tabular_dim):
        super().__init__()
        self.image_encoder = image_encoder()
        self.tabular_encoder = nn.Sequential(nn.Linear(tabular_dim, 64), nn.ReLU(inplace=True), nn.Dropout(0.2), nn.Linear(64, 32), nn.ReLU(inplace=True))
        self.classifier = nn.Sequential(nn.Linear(64 + 32, 48), nn.ReLU(inplace=True), nn.Dropout(0.3), nn.Linear(48, 3))

    def forward(self, images, tabular):
        return self.classifier(torch.cat([self.image_encoder(images), self.tabular_encoder(tabular)], dim=1))


class ResNet18Head(nn.Module):
    def __init__(self):
        super().__init__()
        from torchvision.models import ResNet18_Weights, resnet18
        self.net = resnet18(weights=ResNet18_Weights.DEFAULT)
        self.net.fc = nn.Linear(self.net.fc.in_features, 3)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, images, tabular=None):
        return self.net((images.repeat(1, 3, 1, 1) - self.mean) / self.std)

Training: Adam, cross-entropy, early stopping on validation macro F1 (patience 5), keeping the best checkpoint; the test set is used once.

In [8]:
def evaluate(model, loader):
    model.eval()
    true, pred = [], []
    with torch.no_grad():
        for images, tabular, labels in loader:
            pred += model(images.to(device), tabular.to(device)).argmax(dim=1).cpu().tolist()
            true += labels.tolist()
    return {"Accuracy": accuracy_score(true, pred), "Balanced accuracy": balanced_accuracy_score(true, pred),
            "Macro F1": f1_score(true, pred, average="macro", zero_division=0)}


def train(model, optimizer, loaders, max_epochs, patience=5):
    criterion, best, best_state, wait = nn.CrossEntropyLoss(), -1.0, None, 0
    for epoch in range(max_epochs):
        model.train()
        for images, tabular, labels in loaders["train"]:
            optimizer.zero_grad(set_to_none=True)
            criterion(model(images.to(device), tabular.to(device)), labels.to(device)).backward()
            optimizer.step()
        score = evaluate(model, loaders["validation"])["Macro F1"]
        best, best_state, wait = (score, copy.deepcopy(model.state_dict()), 0) if score > best else (best, best_state, wait + 1)
        if wait >= patience:
            break
    model.load_state_dict(best_state)
    return best


def loaders():
    torch.manual_seed(42)
    return {name: DataLoader(CorneaDataset(part, Z[name].toarray() if hasattr(Z[name], "toarray") else Z[name]),
                             batch_size=16, shuffle=name == "train", num_workers=0) for name, part in parts.items()}

Train and test the three networks (only when `TRAIN_NEURAL_MODELS = True`).

In [9]:
if TRAIN_NEURAL_MODELS:
    random.seed(42), np.random.seed(42), torch.manual_seed(42), torch.cuda.manual_seed_all(42)
    for name, model in [("Image-only CNN", CorneaCNN()), ("Fusion network (image + tabular)", FusionNet(Z["train"].shape[1]))]:
        model = model.to(device)
        best_val = train(model, torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4), loaders(), max_epochs=30)
        results[name] = evaluate(model, loaders()["test"]) | {"Source": f"trained here (best validation macro F1 {best_val:.3f})"}

    resnet = ResNet18Head().to(device)
    for p in resnet.net.parameters():
        p.requires_grad = False
    for p in resnet.net.fc.parameters():
        p.requires_grad = True
    train(resnet, torch.optim.Adam(resnet.net.fc.parameters(), lr=1e-3, weight_decay=1e-4), loaders(), max_epochs=20)
    for p in resnet.net.layer4.parameters():
        p.requires_grad = True
    best_val = train(resnet, torch.optim.Adam([{"params": resnet.net.layer4.parameters(), "lr": 1e-5},
                                               {"params": resnet.net.fc.parameters(), "lr": 1e-4}], weight_decay=1e-4), loaders(), max_epochs=15)
    results["ResNet-18, layer4 fine-tuned"] = evaluate(resnet, loaders()["test"]) | {"Source": f"trained here (best validation macro F1 {best_val:.3f})"}
else:
    results["Image-only CNN"] = {"Accuracy": 0.5461, "Balanced accuracy": 0.3960, "Macro F1": 0.3706,
                                 "Source": "recorded: images/04 saved output (best validation macro F1 0.417)"}
    results["Fusion network (image + tabular)"] = {"Accuracy": 0.8289, "Balanced accuracy": 0.8086, "Macro F1": 0.8179,
                                                   "Source": "recorded: images/03 saved output (best validation macro F1 0.807)"}
    results["ResNet-18, layer4 fine-tuned"] = {"Accuracy": np.nan, "Balanced accuracy": np.nan, "Macro F1": np.nan,
                                               "Source": "recorded: images/02 saved output, validation only (macro F1 0.418; frozen head 0.394)"}

## 3. Results on the shared test set (152 tissues, single split, no uncertainty)

In [10]:
table = pd.DataFrame(results).T[["Macro F1", "Balanced accuracy", "Accuracy", "Source"]]
table[["Macro F1", "Balanced accuracy", "Accuracy"]] = table[["Macro F1", "Balanced accuracy", "Accuracy"]].astype(float).round(3)
table

,Macro F1,Balanced accuracy,Accuracy,Source
Tabular Random Forest,0.906,0.893,0.901,computed here
Image-only CNN,0.371,0.396,0.546,recorded: images/04 saved output (best validat...
Fusion network (image + tabular),0.818,0.809,0.829,recorded: images/03 saved output (best validat...
"ResNet-18, layer4 fine-tuned",NaN,NaN,NaN,"recorded: images/02 saved output, validation o..."


## Interpretation (superseded)

- The tabular Random Forest scored 0.906 macro F1 on this test set, the fusion network 0.818 and the image-only CNN 0.371, which recognised only 1 of 25 grade B tissues. Other CNN variants in images/02 (class weights, 448-pixel input, partly fine-tuned ResNet-18) reached only 0.33–0.48 on validation.
- This shows that **these configurations** failed, not that images carry no information. The fusion result mixes two changes (adding images and switching from a Random Forest to a small neural network), so it cannot isolate the value of the image.
- None of these numbers is a performance estimate: fellow eyes leak across partitions, and one 152-tissue test set has wide, unquantified uncertainty.
- **What replaced it:** frozen full-resolution ResNet-50 embeddings with PCA, the same Random Forest with and without images, and donor-grouped 5 × 5 CV with paired tests (Hybrid 01–04). Under that design the image adds no significant benefit (Hybrid 03).